### Classifier

**Clasiffier used as beta version until we get a more defined output in the resnet50 and pooling processs. Those are in 2D and 3D.**


In [14]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import models
import import_ipynb
import nbformat
import ast
import pydicom
import glob


**Importing report classification rules**

In [15]:
import ast

with open("report_extractor.ipynb", "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

for cell in nb.cells:

    if cell.cell_type != "code":
        continue

    # Ignore cells containing Jupyter magic commands
    if cell.source.lstrip().startswith("%"):
        continue

    try:
        tree = ast.parse(cell.source)
    except SyntaxError:
        continue

    for node in tree.body:

        # Extract only the _near function
        if isinstance(node, ast.FunctionDef) and node.name == "_near":

            exec(
                compile(
                    ast.Module(
                        body=[node],
                        type_ignores=[]
                    ),
                    "<_near>",
                    "exec"
                )
            )

        # Extract only the RULES dictionary
        elif isinstance(node, ast.Assign):

            for target in node.targets:

                if (
                    isinstance(target, ast.Name)
                    and target.id == "RULES"
                ):

                    exec(
                        compile(
                            ast.Module(
                                body=[node],
                                type_ignores=[]
                            ),
                            "<RULES>",
                            "exec"
                        )
                    )

In [16]:
print(_near)
print(RULES.keys())


<function _near at 0x70fa08cc27a0>
dict_keys(['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'])


**Classifying medical reports**

In [17]:
def classify_report(report):
    """
    Classify a single report using the existing RULES and _near function.
    """

    tokens = report.split()

    result = {}

    for label, (anchor, evidence) in RULES.items():

        if evidence:
            result[label] = _near(
                tokens,
                anchor,
                evidence
            )
        else:
            result[label] = int(anchor in tokens)

    return result

**Defining classification targets**

In [18]:
finding_cols = [
    'ACL',
    'MCL',
    'Medial Meniscus',
    'Lateral Meniscus',
    'Medial OA',
    'Lateral OA',
    'PF OA',
    'Effusion',
    'Synovitis',
    "Baker's",
    'Contusion',
    'Fracture'
]

**Loading labeled training data**

In [19]:
train = pd.read_csv("../data/train.csv")
gold = train[
    train[finding_cols].notna().all(axis=1)
]

print("The labeled part of the data has the shape:", gold.shape)

The labeled part of the data has the shape: (58, 14)


**Working with "Gold" reports**

In [20]:
y = gold[finding_cols].astype(np.float32).values

print(y.shape)

(58, 12)


In [21]:
study_uids = gold["StudyInstanceUID"].astype(str).values

In [22]:
label_lookup = (gold.set_index("StudyInstanceUID")[finding_cols].to_dict("index"))

In [23]:
test_uid = gold["StudyInstanceUID"].iloc[48]

print(test_uid)
print(label_lookup[test_uid])

1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995
{'ACL': 0.0, 'MCL': 0.0, 'Medial Meniscus': 1.0, 'Lateral Meniscus': 0.0, 'Medial OA': 0.0, 'Lateral OA': 0.0, 'PF OA': 1.0, 'Effusion': 1.0, 'Synovitis': 1.0, "Baker's": 0.0, 'Contusion': 0.0, 'Fracture': 0.0}


**Match DICOM files to their StudyInstanceUID**

In [24]:
# ====================================================================================================
# Match DICOM files to their StudyInstanceUID
# ====================================================================================================

SAMPLES_DIR = "../data/samples"

# Discover all DICOM files recursively
dcm_files = (
    glob.glob(os.path.join(SAMPLES_DIR, "**", "*.dcm"), recursive=True)
    + glob.glob(os.path.join(SAMPLES_DIR, "**", "*.DCM"), recursive=True)
)

dicom_studies = []

for file_path in dcm_files:
    try:
        dcm_header = pydicom.dcmread(
            file_path,
            stop_before_pixels=True
        )

        study_uid = str(dcm_header.get("StudyInstanceUID", ""))

        if study_uid:
            dicom_studies.append({
                "StudyInstanceUID": study_uid,
                "image_path": file_path
            })

    except Exception as e:
        print(f"Skipping unreadable file {file_path}: {e}")

dicom_studies = pd.DataFrame(dicom_studies)

### 2D Classifier

In [ ]:
inputs = tf.keras.Input(
    shape=(2048,),
    name="study_features")


# 12 independent Logistic Regression classifiers
# Each sigmoid output represents the probability of one pathology


outputs = layers.Dense(
    12,
    activation="sigmoid",
    name="pathology_predictions")(inputs)



# Build classifier model

classifier = models.Model(
    inputs=inputs,
    outputs=outputs,
    name="Knee_MultiLabel_Logistic_Classifier")


classifier.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(
            name="auc",
            multi_label=True)])




### 3D Classifier

In [ ]:
inputs = tf.keras.Input(
    shape=(None, 2048),
    name="study_slice_features")

attention_scores = layers.Dense(
    1,
    activation="tanh",
    name="attention_scores")(inputs)


attention_weights = layers.Softmax(
    axis=1,
    name="attention_weights")(attention_scores)



weighted_features = layers.Multiply(
    name="weighted_slice_features"
)([
    inputs,
    attention_weights])


study_features = layers.Lambda(
    lambda x: tf.reduce_sum(x, axis=1),
    name="attention_pooling"
)(weighted_features)


# 12 independent Logistic Regression classifiers

outputs = layers.Dense(    12,
    activation="sigmoid",
    name="pathology_predictions")(study_features)



classifier_3d = models.Model(
    inputs=inputs,
    outputs=outputs,
    name="Knee_3D_Attention_Logistic_Classifier")



classifier_3d.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(
            name="auc",
            multi_label=True)])